In [17]:
import pdfplumber

file_path = r'C:\Users\fomichevva4\Desktop\Проекты\2026-05-12 Робот ДОМ РФ\pdf\0001_Жилой комплекс_Саларьево парк__94305491_obj68264-pd77-003252.pdf'
with pdfplumber.open(file_path) as pdf:
    page = pdf.pages[174]
    table = page.extract_table()
    for row in table:
        print(row)
    print('-----------------------------------------------------')
    alletters = []
    for row in table:
        for text in row:
            alletters += text
    singletext = ''.join(alletters)
    print(f'Наш текст следующий: {singletext}')

    #Найдем текст "Общая площадь объекта"
    text_pattern = "Об щая площ адь объ ект а:"

['', '9.2.12', 'Ли тера:']
['', '9.2.13', 'Кор пус:']
['', '9.2.14', 'Стро ение:']
['', '9.2.15', 'Вла дение:']
['', '9.2.16', 'Блок-сек ция:']
['', '9.2.17', 'Уточн ен ие адр е са:\nГ.Моск ва, НАО, вн.тер.г. му ницип аль ный ок руг Ком му нар ка, ули -\nца Са ларь ев ская, зем ель ный учас ток 17, кор пус 82']
['', '9.2.18', 'Наз нач ение объ ект а:\nЖи лое']
['', '9.2.19', 'Ми нимальн ое кол-во этаж ей:\n2']
['', '9.2.20', 'Макс им альн ое кол-во этаж ей:\n25']
['', '9.2.21', 'Об щая площ адь объ ект а:\n100711.4 м2']
['', '9.2.22', 'Мат ериа л на ружн ых стен и карк ас а объ ект а:\nИной вид мат ериа лов нар ужн ых стен и кар ка сов (Мат ериа л на-\nружн ых стен - на весн ая пан ель с обл и цов кой из плитк и и нав ес -\nной вен тил ируе мый фас ад с обл и цов кой из плитк и. Кар кас - мо -\nнолит ный жел езо бетонн ый.)']
['', '9.2.23', 'Мат ериа л пе рек ры тий:\nМо нолит ные жел езо бетонн ые']
['', '9.2.24', 'Класс энер гет ическ ой эфф ективн ости:\nA+']
['9.3 О сумм е об щей п

In [18]:
import re
import pdfplumber

file_path = r'C:\Users\fomichevva4\Desktop\Проекты\2026-05-12 Робот ДОМ РФ\pdf\0001_Жилой комплекс_Саларьево парк__94305491_obj68264-pd77-003252.pdf'
# В pdfplumber нумерация страниц с 0.
# То есть pages[174] — это 175-я страница PDF.
PAGE_INDEX = 174
def make_fuzzy_word(word: str) -> str:
    """
    Делает regex для слова, допускающий пробелы внутри слова.
    Например:
    объект -> о\\s*б\\s*ъ\\s*е\\s*к\\s*т
    Поэтому regex найдет и:
    объект
    объ ект
    о б ъ е к т
    """
    separators = r"[\s\|\u00a0]*"
    return separators.join(re.escape(ch) for ch in word)
def make_fuzzy_phrase(phrase: str) -> str:
    """
    Делает regex для фразы, допускающий:
    - пробелы внутри слов;
    - переносы строк;
    - разделители таблиц;
    - дефисы;
    - двоеточия;
    - точки с запятой.
    """
    words = phrase.lower().split()
    word_separator = r"[\s\|\u00a0:;,\.\-–—]*"
    return word_separator.join(make_fuzzy_word(word) for word in words)
def normalize_text(text: str) -> str:
    text = text or ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()
def get_context(text: str, start: int, end: int, chars: int = 250) -> str:
    left = max(0, start - chars)
    right = min(len(text), end + chars)
    fragment = text[left:right]
    fragment = normalize_text(fragment)
    return fragment
# Ищем именно фразу "общая площадь объекта"
label_pattern = make_fuzzy_phrase("общая площадь объекта")
# Ищем фразу + число после нее
area_pattern = re.compile(
    rf"{label_pattern}"
    rf"[\s\|\u00a0:;,\.\-–—]*"
    rf"(?P<area>\d[\d\s.,]*)"
    rf"\s*(?P<unit>кв\.?\s*м|м2|м²)?",
    flags=re.IGNORECASE | re.DOTALL,
)
# Отдельно regex только для обнаружения самой фразы без площади
label_only_pattern = re.compile(
    label_pattern,
    flags=re.IGNORECASE | re.DOTALL,
)
# Отдельные слова — на случай, если нужно проверить наличие каждого слова отдельно
separate_words_patterns = {
    "общая": re.compile(make_fuzzy_word("общая"), flags=re.IGNORECASE | re.DOTALL),
    "площадь": re.compile(make_fuzzy_word("площадь"), flags=re.IGNORECASE | re.DOTALL),
    "объекта": re.compile(make_fuzzy_word("объекта"), flags=re.IGNORECASE | re.DOTALL),
}
with pdfplumber.open(file_path) as pdf:
    page = pdf.pages[PAGE_INDEX]
    table = page.extract_table()
    if not table:
        print("Таблица на странице не найдена.")
    else:
        print("ТАБЛИЦА:")
        print("-----------------------------------------------------")
        for row in table:
            print(row)
        print("-----------------------------------------------------")
        # Собираем текст из таблицы.
        # Важно: cell может быть None, поэтому делаем cell or "".
        all_cells = []
        for row in table:
            for cell in row:
                if cell:
                    all_cells.append(cell)
        # Разделитель | помогает видеть границы ячеек таблицы
        table_text = " | ".join(all_cells)
        print()
        print("НАШ ТЕКСТ СЛЕДУЮЩИЙ:")
        print(table_text)
        print()
        print("-----------------------------------------------------")
        print("ПОИСК ФРАЗЫ: 'общая площадь объекта'")
        print("-----------------------------------------------------")
        # 1. Ищем фразу + значение площади
        matches = list(area_pattern.finditer(table_text))
        if matches:
            print(f"Найдено совпадений с площадью: {len(matches)}")
            for i, match in enumerate(matches, start=1):
                area = normalize_text(match.group
("area"))
                unit = normalize_text(match.group
("unit") or "")
                print()
                print(f"Совпадение №{i}")
                print(f"Найденная площадь: {area} {unit}".strip())
                print("Контекст:")
                print(get_context(table_text, match.start(), match.end()))
        else:
            print("Фраза с числом площади не найдена.")
            # 2. Ищем хотя бы саму фразу без числа
            label_matches = list(label_only_pattern.finditer(table_text))
            if label_matches:
                print()
                print(f"Но сама фраза 'общая площадь объекта' найдена: {len(label_matches)} раз.")
                for i, match in enumerate(label_matches, start=1):
                    print()
                    print(f"Фраза №{i}")
                    print("Контекст:")
                    print(get_context(table_text, match.start(), match.end()))
            else:
                print()
                print("Целиком фраза 'общая площадь объекта' не найдена.")
                # 3. Проверяем отдельные слова
                print()
                print("Проверка отдельных слов:")
                for word, pattern in separate_words_patterns.items():
                    word_matches = list(pattern.finditer(table_text))
                    print(f"Слово '{word}': найдено {len(word_matches)} раз")
                    for i, match in enumerate(word_matches[:5], start=1):
                        print(f"  {word} №{i}: {get_context(table_text, match.start(), match.end(), chars=80)}")
print("====================================")
print(area)
print("====================================")

ТАБЛИЦА:
-----------------------------------------------------
['', '9.2.12', 'Ли тера:']
['', '9.2.13', 'Кор пус:']
['', '9.2.14', 'Стро ение:']
['', '9.2.15', 'Вла дение:']
['', '9.2.16', 'Блок-сек ция:']
['', '9.2.17', 'Уточн ен ие адр е са:\nГ.Моск ва, НАО, вн.тер.г. му ницип аль ный ок руг Ком му нар ка, ули -\nца Са ларь ев ская, зем ель ный учас ток 17, кор пус 82']
['', '9.2.18', 'Наз нач ение объ ект а:\nЖи лое']
['', '9.2.19', 'Ми нимальн ое кол-во этаж ей:\n2']
['', '9.2.20', 'Макс им альн ое кол-во этаж ей:\n25']
['', '9.2.21', 'Об щая площ адь объ ект а:\n100711.4 м2']
['', '9.2.22', 'Мат ериа л на ружн ых стен и карк ас а объ ект а:\nИной вид мат ериа лов нар ужн ых стен и кар ка сов (Мат ериа л на-\nружн ых стен - на весн ая пан ель с обл и цов кой из плитк и и нав ес -\nной вен тил ируе мый фас ад с обл и цов кой из плитк и. Кар кас - мо -\nнолит ный жел езо бетонн ый.)']
['', '9.2.23', 'Мат ериа л пе рек ры тий:\nМо нолит ные жел езо бетонн ые']
['', '9.2.24', 'Класс э

In [19]:
make_fuzzy_phrase('hello world')

'h[\\s\\|\\u00a0]*e[\\s\\|\\u00a0]*l[\\s\\|\\u00a0]*l[\\s\\|\\u00a0]*o[\\s\\|\\u00a0:;,\\.\\-–—]*w[\\s\\|\\u00a0]*o[\\s\\|\\u00a0]*r[\\s\\|\\u00a0]*l[\\s\\|\\u00a0]*d'

In [20]:
res = make_fuzzy_word('hello')
print(res)

h[\s\|\u00a0]*e[\s\|\u00a0]*l[\s\|\u00a0]*l[\s\|\u00a0]*o


In [43]:
area_pattern = re.compile(
    rf"{label_pattern}"
    rf"[\s\|\u00a0:;,\.\-–—]*"
    rf"(?P<area>\d[\d\s.,]*)"
    rf"\s*(?P<unit>кв\.?\s*м|м2|м²)?",
    flags=re.IGNORECASE | re.DOTALL,
)
text = "общая площадь объекта: 123,5 кв.м"
match = area_pattern.search(text)
if match:
    print(match.groupdict())
    print("=================")
    print(area_pattern.groupindex)
    print(match.group('area'))
    print(match.group('unit'))

{'area': '123,5 ', 'unit': 'кв.м'}
{'area': 1, 'unit': 2}
123,5 
кв.м


In [ ]:
# Главное исправление по твоей строке
# Ты написал:
# text_pattern = "Об щая площ адь объ ект а:"
# Так искать плохо, потому что в другом PDF может быть:
# Общая площадь объекта:
# или:
# Общая
# площадь
# объекта
# или:
# Общая | площадь | объекта
# Поэтому лучше не писать вручную:
# "Об щая площ адь объ ект а:"
# а генерировать гибкий regex:
# label_pattern = make_fuzzy_phrase("общая площадь объекта")
# Он будет искать все такие варианты.


# Если нужно искать не только «общая площадь объекта», но и похожие формулировки
# Можно добавить список:
# phrases = [
#     "общая площадь объекта",
#     "общая площадь здания",
#     "общая площадь объекта капитального строительства",
#     "площадь объекта",
#     "площадь здания",
# ]
# И потом прогонять каждую фразу через make_fuzzy_phrase().